# Re-do scVI Integration and re-clustering

In [ ]:
# !pip install --quiet scvi-tools==1.2.1 scanpy leidenalg igraph
# %cd <project>/notebooks

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Config

In [ ]:
run = "S11_with_AN"                   

out_dir = "./../snRNAseq/output/cellbender/cleanup"

presets = {
    "S08_clean": dict(
        inputs=[dict(name="main",
                     path="./../snRNAseq/output/cellbender/S7_scVI_integrated_adata_cluster_latent10_logreg_fine_cell_type_clean.h5ad",
                     samples=None)],
        merged_path=None,
        output_path=f"{out_dir}/S9_scVI_integrated_adata_clean_latent10.h5ad",
        scvi_output_dir=f"{out_dir}/scvi_results_S08",
        figures_output_dir="./../figures-cellbender/integration/1_scvi/1_clean",
        batch_key="donor",
        final_n_neighbors=20, leiden_resolution=0.5, umap_spread=1.0,
    ),
    "S11_with_AN": dict(
        inputs=[
            dict(name="main",
                 path=f"{out_dir}/S10c_cellbend_cluster_latent10_cluster_major_cell_type.h5ad",
                 samples=None),
            # second cohort (lung06-AN): keep adjacent-normal samples only; its tumour samples are already in "main"
            dict(name="AN_cohort",
                 path="./../../lung06-AN/snRNAseq/output/cellbender/cleanup/S10c_cellbend_cluster_latent10_cluster_major_cell_type.h5ad",
                 samples=["A074LCGAN", "A111LCGAN", "A150LCGAN", "A157LCGAN", "A3944LCGAN", "A4177LCGAN", "A4243LCGAN","A161LCGA", "A3808LCGAN"]), 
                 # Add normal adjacent samples which has been preprocessed and clean up similarly
        ],
        merged_path=f"{out_dir}/S10d_merged_11062025.h5ad",   # merged counts, input to the GPU step
        output_path=f"{out_dir}/S11_AN_all_scvi_adata_merge.h5ad",
        scvi_output_dir=f"{out_dir}/scvi_results_S11",
        figures_output_dir="./../figures-cellbender/integration/1_scvi/2_AN_merge",
        batch_key="ID",                  # patient ID; paired AN/tumour samples share it
        final_n_neighbors=40, leiden_resolution=0.3, umap_spread=1.5,
    ),
}
cfg = presets[run]
seed = 0

# --- merge ---
sample_col = "sample"
source_col = "source"                   # input name per nucleus (does not overwrite obs["batch"])

# --- scVI (all genes, raw counts) ---
run_training = True                     # False -> load latents saved in scvi_output_dir
counts_layer = "raw_counts"
categorical_covariate_keys = ["batchseq"]
continuous_covariate_keys = ["percent_mito"]
n_latent_values = [10, 20, 30]
latent_prefix = "X_scVI_n_latent_"      # obsm key = latent_prefix + n_latent

# --- exploration (UMAP only; ~4 min per UMAP on 200k nuclei) ---
explore_latents = True                  # compare n_latent_values at the final n_neighbors
explore_neighbors = True                # compare n_neighbors_values on the final latent
n_neighbors_values = [10, 20, 30, 40]
explore_color = "cell_type"

# --- final graph and clusters ---
final_n_latent = 10
umap_min_dist = 0.5
cluster_key = "leiden_scvi"
qc_plot_cols = [cluster_key, source_col, "sample", "ID", "batchseq", "group2", "n_genes",
                "scrublet_score", "percent_mito", "phase", "cell_type"]

for d in (cfg["scvi_output_dir"], cfg["figures_output_dir"]):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = cfg["figures_output_dir"]
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80)
scvi.settings.seed = seed
print(f"run: {run} | scvi-tools {scvi.__version__}")

## 1. Load and merge inputs

In [ ]:
def load_input(name, path, samples=None):
    a = sc.read(path)
    if samples is not None:
        missing = set(samples) - set(a.obs[sample_col])
        assert not missing, f"{name}: samples not found: {sorted(missing)}"
        a = a[a.obs[sample_col].isin(samples)].copy()
    a.obs[source_col] = name
    for slot in (a.obsm, a.obsp, a.varm):
        slot.clear()
    a.uns = {}
    print(f"{name}: {a.n_obs} nuclei, {a.obs[sample_col].nunique()} samples")
    return a


parts = [load_input(**spec) for spec in cfg["inputs"]]

if len(parts) > 1:
    seen = {}
    for p in parts:
        for s in p.obs[sample_col].unique():
            assert s not in seen, f"sample {s} in both {seen[s]} and {p.obs[source_col].iloc[0]}"
            seen[s] = p.obs[source_col].iloc[0]
    assert all(p.var_names.equals(parts[0].var_names) for p in parts), "gene sets differ between inputs"
    adata = ad.concat(parts, join="outer", merge="same", index_unique=None)
    assert adata.obs_names.is_unique, "duplicate barcodes across inputs"
else:
    adata = parts[0]
del parts

adata.obs[source_col] = adata.obs[source_col].astype("category")
if counts_layer not in adata.layers:
    adata.layers[counts_layer] = adata.X.copy()
vals = adata.layers[counts_layer][:1000].data
assert np.allclose(vals, np.round(vals)), f"layers['{counts_layer}'] must hold raw counts"

if cfg["merged_path"]:
    adata.write(cfg["merged_path"])

print(adata.obs[[cfg["batch_key"]] + categorical_covariate_keys].nunique())
pd.crosstab(adata.obs[sample_col], adata.obs[source_col]).loc[lambda d: d.sum(1) > 0]

## 2. Train scVI

In [ ]:
latent_csv = lambda n: os.path.join(cfg["scvi_output_dir"], f"scVI_latent_n_{n}_allGenes.csv")

if run_training:
    scvi.model.SCVI.setup_anndata(
        adata, layer=counts_layer, batch_key=cfg["batch_key"],
        categorical_covariate_keys=categorical_covariate_keys,
        continuous_covariate_keys=continuous_covariate_keys,
    )
    for n in n_latent_values:
        print(f"training scVI, n_latent = {n}")
        model = scvi.model.SCVI(adata, n_latent=n)
        model.train()
        adata.obsm[f"{latent_prefix}{n}"] = model.get_latent_representation()
        pd.DataFrame(adata.obsm[f"{latent_prefix}{n}"], index=adata.obs_names).to_csv(latent_csv(n))
        model.save(os.path.join(cfg["scvi_output_dir"], f"scVI_model_n_latent_{n}"), overwrite=True)
else:
    for n in n_latent_values:
        latent = pd.read_csv(latent_csv(n), index_col=0)
        if not latent.index.equals(adata.obs_names):
            raise ValueError(f"barcode mismatch between {latent_csv(n)} and the AnnData object")
        adata.obsm[f"{latent_prefix}{n}"] = latent.values

[k for k in adata.obsm if k.startswith(latent_prefix)]

## 3. Choose latent size and neighbourhood size

Exploration only

In [ ]:
def umap_grid(adata, settings, color, fname):
    """One UMAP per (title, use_rep, n_neighbors) setting, coloured by `color`."""
    tmp = sc.AnnData(obs=adata.obs[[color]].copy(), obsm={k: adata.obsm[k] for k in {s[1] for s in settings}})
    fig, axs = plt.subplots(1, len(settings), figsize=(5 * len(settings), 5))
    for ax, (title, rep, k) in zip(np.atleast_1d(axs), settings):
        sc.pp.neighbors(tmp, n_neighbors=k, use_rep=rep, random_state=seed)
        sc.tl.umap(tmp, min_dist=umap_min_dist, spread=cfg["umap_spread"], random_state=seed)
        sc.pl.umap(tmp, color=color, ax=ax, show=False, title=title, legend_loc=None, size=2)
    plt.tight_layout()
    plt.savefig(f"{cfg['figures_output_dir']}/{fname}", bbox_inches="tight", dpi=150)
    plt.show()


if explore_latents:
    umap_grid(adata, [(f"n_latent = {n}", f"{latent_prefix}{n}", cfg["final_n_neighbors"]) for n in n_latent_values],
              explore_color, "umap_compare_n_latent.pdf")
if explore_neighbors:
    umap_grid(adata, [(f"n_neighbors = {k}", f"{latent_prefix}{final_n_latent}", k) for k in n_neighbors_values],
              explore_color, f"umap_compare_n_neighbors_latent{final_n_latent}.pdf")

## 4. Final graph, UMAP and Leiden clusters

In [ ]:
final_rep = f"{latent_prefix}{final_n_latent}"
sc.pp.neighbors(adata, n_neighbors=cfg["final_n_neighbors"], use_rep=final_rep, random_state=seed)
sc.tl.umap(adata, min_dist=umap_min_dist, spread=cfg["umap_spread"], random_state=seed)
sc.tl.leiden(adata, key_added=cluster_key, resolution=cfg["leiden_resolution"],
             flavor="leidenalg", random_state=seed)

# log-normalised values only for plotting; X is reset to raw counts before saving
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

cols = [c for c in qc_plot_cols if c in adata.obs and not (c == source_col and adata.obs[c].nunique() < 2)]
sc.pl.umap(adata, color=cols, color_map="OrRd", ncols=4, wspace=0.3, size=10,
           save=f"_scVI_latent{final_n_latent}.pdf")
sc.pl.umap(adata, color=cluster_key, legend_loc="on data", legend_fontsize="xx-small", size=4,
           save=f"_scVI_latent{final_n_latent}_clustersOnData.pdf")
pd.crosstab(adata.obs[cluster_key], adata.obs["cell_type"])

## 5. Save (X = raw counts)

In [ ]:
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(cfg["output_path"])
adata